In [1]:
!pip install networkx python-louvain

 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 4.5 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for python-louvain: filename=python_louvain-0.16-py3-none-any.whl size=9460 sha256=82b86934f7e128ec7e639e156975a1e660fb055dad978f4b369fe6f1324b2ca9
  Stored in directory: /home/marcelly/.cache/pip/wheels/40/f1/e3/485b698c520fa0baee1d07897abc7b8d6479b7d199ce96f4af
Successfully built python-louvain


In [ ]:
#--BUILD k-NN GRAPH WITH SelectedClean_nhis2023.csv--#

In [4]:
#Normalize AGEP_A between [0-1]
import pandas as pd
import numpy as np

df = pd.read_csv("SelectedClean_nhis2023.csv")

AGE_COL = "AGEP_A"
#compute min/max 
age_min = float(df[AGE_COL].min())
age_max = float(df[AGE_COL].max())

def add_scaled_age(df_in, age_col=AGE_COL, age_min=age_min, age_max=age_max, out_col=None):
    if out_col is None:
        out_col = f"{age_col}_scaled"
    if pd.isna(age_min) or pd.isna(age_max) or age_max <= age_min:
        df_in[out_col] = 0.0  # degenerate case: avoid divide-by-zero
    else:
        df_in[out_col] = (df_in[age_col] - age_min) / (age_max - age_min)
    return df_in

#create a working copy 
df_scaled_age = add_scaled_age(df.copy(), age_col=AGE_COL, age_min=age_min, age_max=age_max)

print(df_scaled_age.columns.tolist())             
print(df_scaled_age[['AGEP_A','AGEP_A_scaled']].head())




['HISPALLP_A', 'SEX_A', 'AGEP_A', 'ANXEV_A', 'AGEP_A_scaled']
   AGEP_A  AGEP_A_scaled
0      67       0.604938
1      73       0.679012
2      48       0.370370
3      42       0.296296
4      50       0.395062


In [7]:
##Checking for best k 
import numpy as np
import pandas as pd
from sklearn.neighbors import kneighbors_graph
from scipy.sparse import csr_matrix, csgraph

def sweep_k_for_knn_age(
    df_scaled: pd.DataFrame,
    scaled_col: str = "AGEP_A_scaled",
    ks = tuple(range(3, 31)),     # k values to try
    target_gcc: float = 0.98,     # target giant component coverage
    dtype=np.float32,
    compute_weights: bool = True,  # Cmpute RBF sigma & mean similarity
    age_col: str | None = "AGEP_A",
    age_min: float | None = None,
    age_max: float | None = None,
    verbose: bool = True,
):

    X = df_scaled[[scaled_col]].to_numpy(dtype=dtype, copy=False)
    n = X.shape[0]
    results = []

    for k in ks:
        k_eff = min(max(1, k), max(1, n-1))  # guard

        # kNN distances (Euclidean in 1D), directed 
        A_dist = kneighbors_graph(
            X, n_neighbors=k_eff, mode="distance",
            metric="euclidean", include_self=False, n_jobs=-1
        ).tocsr().astype(dtype)

        # unweighted, undirected union graph for connectivity stats 
        U = (A_dist + A_dist.T)            # union
        U.data[:] = 1.0                    # binarize
        U = U.tocsr()                      # ensure CSR

        # connected components
        n_comp, labels = csgraph.connected_components(U, directed=False)
        # giant component fraction
        if n_comp > 0:
            _, counts = np.unique(labels, return_counts=True)
            gcc_frac = counts.max() / n
        else:
            gcc_frac = 0.0

        # edges & degree
        undirected_edges = U.nnz // 2
        mean_degree = float(U.sum(axis=1).mean())

        row = {
            "k": k_eff,
            "n_components": n_comp,
            "giant_component_frac": gcc_frac,
            "edges_undirected": undirected_edges,
            "mean_degree": mean_degree,
        }

        # convert distances into RBF weights and summarize
        if compute_weights:
            nz = A_dist.data
            sigma = float(np.median(nz)) if nz.size else 1.0
            W = A_dist.tocsr(copy=True)
            W.data = np.exp(-(W.data**2) / (2.0 * sigma**2 + 1e-12)).astype(dtype)
            W = ((W + W.T) * 0.5).tocsr()  # symmetric
            row.update({
                "sigma_median": sigma,
                "mean_similarity": float(W.data.mean()) if W.nnz else 0.0
            })

        results.append(row)

    df_res = pd.DataFrame(results).sort_values("k").reset_index(drop=True)

    
    # pick the smallest k 
    candidates = df_res[df_res["giant_component_frac"] >= target_gcc]
    if not candidates.empty:
        # keep mean_degree below a soft upper bound (e.g., <= 25)
        soft_cap = candidates[candidates["mean_degree"] <= 25]
        if not soft_cap.empty:
            recommended_k = int(soft_cap.iloc[0]["k"])
        else:
            recommended_k = int(candidates.iloc[0]["k"])
    else:
        # the k with the max giant component fraction; break ties by smaller k
        best_row = df_res.sort_values(
            ["giant_component_frac", "k"], ascending=[False, True]
        ).iloc[0]
        recommended_k = int(best_row["k"])

    sigma_star = None
    if compute_weights and "sigma_median" in df_res.columns:
        sigma_star = float(df_res.loc[df_res["k"] == recommended_k, "sigma_median"].iloc[0])

    # figure out the age bounds for the printout
    used_age_min = age_min
    used_age_max = age_max
    if (used_age_min is None or used_age_max is None) and age_col and (age_col in df_scaled.columns):
        used_age_min = float(df_scaled[age_col].min())
        used_age_max = float(df_scaled[age_col].max())


    if verbose:
        print(f"Recommended k: {recommended_k}")
        if sigma_star is not None:
            print(f"Sigma* (median kNN distance at k={recommended_k}): {sigma_star:.6f}")
        if used_age_min is not None and used_age_max is not None:
            print(f"Age scaling used: min={used_age_min:.3f}, max={used_age_max:.3f}")
        else:
            # fall back to scaled column range (if original min/max not provided)
            smin = float(df_scaled[scaled_col].min())
            smax = float(df_scaled[scaled_col].max())
            print(f"Scaled range for '{scaled_col}': [{smin:.6f}, {smax:.6f}]")

    return df_res, recommended_k


# Ensure df_scaled_age already has AGEP_A_scaled ∈ [0,1]
df_metrics, k_star = sweep_k_for_knn_age(
    df_scaled_age,
    scaled_col="AGEP_A_scaled",
    ks=range(5, 31),
    target_gcc=0.98,
    compute_weights=True,
    age_col="AGEP_A",          # so it can print original age bounds
    age_min=float(df_scaled_age["AGEP_A"].min()),  # optional but precise
    age_max=float(df_scaled_age["AGEP_A"].max()),
    verbose=True
)

print(df_metrics.head())
print("Recommended k:", k_star)

Recommended k: 30
Sigma* (median kNN distance at k=30): 0.000000
Age scaling used: min=18.000, max=99.000
   k  n_components  giant_component_frac  edges_undirected  mean_degree  \
0  5         29460              0.000034                 0     0.000000   
1  6         29454              0.000238                 6     0.000407   
2  7         29453              0.000272                12     0.000815   
3  8         29452              0.000305                18     0.001222   
4  9         29451              0.000339                24     0.001629   

   sigma_median  mean_similarity  
0           0.0         0.503590  
1           0.0         0.504176  
2           0.0         0.504766  
3           0.0         0.505359  
4           0.0         0.505954  
Recommended k: 30


In [8]:
#Building k-NN graph based on AGE only

# k-NN graph from AGE only (no protected attrs), RBF similarity on age
import json
import numpy as np
from sklearn.neighbors import kneighbors_graph #memory-efficient k-NN graph builder
from scipy import sparse

def build_knn_from_pre_scaled(
    df_scaled, scaled_col="AGEP_A_scaled",
    k=30, #number of nearest neighbors per node ( Recommended k)
    sigma=None, #width of the RBF/Gaussian kernel estimated from data
    mutual=False, #keep neighbors based on union (average the two directions).
    save_prefix="A_graph_age_rbf",  # name of output graph
    dtype=np.float32,# saving memory
    sigma_floor=1e-4  # small lower bound in scaled-age units
):
    X = df_scaled[[scaled_col]].to_numpy(dtype=dtype, copy=False) # array containing only pre-scaled age without unnecessary copies
    n = X.shape[0] # number of nodes
    if k >= n:
        k = max(1, n-1)

    # kNN distances in 1D (Euclidean)
    A_dist = kneighbors_graph(
        X, n_neighbors=k, mode="distance", # edge values are distances
        metric="euclidean", 
        include_self=False, n_jobs=-1 # avoids self-loops and sppedup with all CPU cores
    ).tocsr().astype(dtype) 

    # choose sigma based on estimate of the median of the non-zero neighbor distances
    if sigma is None:
        nz = A_dist.data
        pos = nz[nz > 0]  # ignore same-age zeros
        if pos.size:
            sigma = float(np.median(pos))
        else:
            # all neighbors at zero distance → pick a small but usable sigma
            sigma = 0.01
    sigma = max(float(sigma), sigma_floor)

    # distance -> RBF similarity, then symmetrize
    A_sim = A_dist.tocsr(copy=True)
    A_sim.data = np.exp(-(A_sim.data**2) / (2.0 * sigma**2 + 1e-12)).astype(dtype) # prevents division by zero if sigma is tiny
    A_sim = ((A_sim + A_sim.T) * 0.5).tocsr() #include edges appearing in either direction and average of the two weights

    # save outputs + minimal params (no age min/max needed because you pre-scaled)
    sparse.save_npz(f"{save_prefix}.npz", A_sim)
    params = {
        "scaled_col": scaled_col,
        "k": int(k),
        "sigma": float(sigma),
        "dtype": "float32",
        "mutual": bool(mutual),
        "similarity": "rbf_on_euclidean_age (pre-scaled)"
    }
    with open(f"{save_prefix}_params.json", "w") as f:
        json.dump(params, f, indent=2)

    print(f"Saved graph: {save_prefix}.npz  (nodes={n}, edges={A_sim.nnz//2} undirected approx.)")
    print(f"Saved params: {save_prefix}_params.json  (k={k}, sigma={sigma:.6f})")
    return A_sim, params
 
A_sim, params = build_knn_from_pre_scaled(
    df_scaled_age,
    scaled_col="AGEP_A_scaled",
    k=30,            
    sigma=None,      
    mutual=False,
    save_prefix="A_graph_age_rbf"
)


Saved graph: A_graph_age_rbf.npz  (nodes=29460, edges=851700 undirected approx.)
Saved params: A_graph_age_rbf_params.json  (k=30, sigma=0.024691)


In [9]:
print("Final sigma used:", params["sigma"])
print("Mean edge similarity:", float(A_sim.data.mean()) if A_sim.nnz else 0.0)

Final sigma used: 0.024691343307495117
Mean edge similarity: 0.5188099145889282


In [ ]:
#Code sources

#https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
#https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.kneighbors_graph.html
#https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csgraph.connected_components.html
#https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.rbf_kernel.html
#https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.kernels.RBF.html
#https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.save_npz.html
#https://www.geeksforgeeks.org/machine-learning/k-nearest-neighbours/
#https://www.datacamp.com/tutorial/k-nearest-neighbor-classification-scikit-learn
#https://www.youtube.com/watch?v=Nz73vXn5afE
#https://www.youtube.com/watch?v=9-sIgMrMPuY
#https://youtu.be/gGdlYicGIIo?si=tElphyYBEvXdA0Tx
#https://www.geeksforgeeks.org/machine-learning/radial-basis-function-kernel-machine-learning/
#https://machinelearningmastery.com/normalize-standardize-time-series-data-python/
#https://www.youtube.com/watch?v=94f_bId5HGI

